# Diabetes Prediction & Healthcare Analytics
### Academic End-to-End Machine Learning Pipeline
---
**Objective:** Predict whether an individual is **Diabetic (`1`)** or **Non-Diabetic (`0`)** using clinical biomarkers and lifestyle risk factors.

> **Medical Disclaimer:** This analysis is conducted strictly for educational and research purposes. It is not intended for clinical diagnostics.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Insert parent path so src package is accessible
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.preprocessing import (
    load_dataset, inspect_dataset, build_preprocessor,
    prepare_features, get_feature_names_out, RANDOM_STATE, TEST_SIZE
)

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries and dependencies loaded successfully.')

## 1. Load and Inspect Dataset
We load `diabetes_binary_health.csv` and inspect dimension properties, missing entries, duplicates, and class balance.

In [ ]:
data_path = Path('data/diabetes_binary_health.csv') if Path('data/diabetes_binary_health.csv').exists() else Path('../data/diabetes_binary_health.csv')
df = load_dataset(data_path)
summary = inspect_dataset(df)

print(f"Dataset Shape: {summary['shape']}")
print(f"Missing Values: {summary['missing_values']}")
print(f"Duplicate Rows: {summary['duplicate_rows']}")
print(f"Target Distribution: {summary['class_distribution']}")
df.head()

## 2. Healthcare Analytics & Exploratory Data Analysis
We examine the density and distributions of continuous biomarkers (Glucose, BMI, Blood Pressure, Cholesterol, Age) across diabetes states.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
palette = {0: '#4C72B0', 1: '#DD8452'}

# Target Class Balance
sns.countplot(data=df, x='Diabetes_binary', palette=palette, ax=axes[0, 0])
axes[0, 0].set_title('Target Class Balance (0: Non-Diabetic, 1: Diabetic)')

# Biomarker distributions
features = ['Age', 'BMI', 'Glucose', 'Blood_Pressure', 'Cholesterol']
for ax, feat in zip(axes.flat[1:], features):
    sns.kdeplot(data=df, x=feat, hue='Diabetes_binary', palette=palette, fill=True, common_norm=False, ax=ax)
    ax.set_title(f'{feat} Distribution by Diabetes Status')

plt.tight_layout()
plt.show()

### Feature Correlation Matrix
> *Note:* Pearson correlation indicates statistical co-variation, not medical causality.

In [ ]:
corr_cols = ['Age', 'BMI', 'Glucose', 'Blood_Pressure', 'Cholesterol', 'Family_History', 'Heart_Disease', 'High_BP', 'High_Chol', 'Diabetes_binary']
corr = df[[c for c in corr_cols if c in df.columns]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=0.5)
plt.title('Health Risk Factor Correlation Heatmap')
plt.show()

## 3. Data Preprocessing & Train-Test Split
We set up a reproducible pipeline where transformers are fitted strictly on the training set to prevent data leakage.

In [ ]:
X, y = prepare_features(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
preprocessor = build_preprocessor()
print(f'Training Samples: {len(X_train)} | Testing Samples: {len(X_test)}')

## 4. Random Forest Classifier Pipeline
We encapsulate preprocessor + RandomForestClassifier into a unified scikit-learn Pipeline.

In [ ]:
from sklearn.pipeline import Pipeline

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_split=5,
        min_samples_leaf=2, max_features='sqrt', class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
print('Random Forest model trained successfully.')

## 5. Model Evaluation
We evaluate generalization performance on the test set across multiple metrics.

In [ ]:
y_pred = rf_pipeline.predict(X_test)
y_prob = rf_pipeline.predict_proba(X_test)[:, 1]

metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-Score': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_prob),
}

for m, val in metrics.items():
    print(f'{m:<12}: {val:.4f}')

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Non-Diabetic', 'Diabetic']))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Diabetic', 'Diabetic'], yticklabels=['Non-Diabetic', 'Diabetic'], ax=ax1)
ax1.set_title('Confusion Matrix')
ax1.set_xlabel('Predicted Label')
ax1.set_ylabel('True Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax2.plot(fpr, tpr, color='#4C72B0', lw=2, label=f"ROC Curve (AUC = {metrics['ROC-AUC']:.4f})")
ax2.plot([0, 1], [0, 1], 'k--', lw=1)
ax2.set_title('ROC Curve')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate (Recall)')
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()

## 6. Feature Importance Interpretation
Mean Decrease in Impurity (MDI) indicates the relative predictive contribution of each feature in the tree ensemble.

In [ ]:
feature_names = get_feature_names_out(rf_pipeline.named_steps['preprocessor'])
importances = rf_pipeline.named_steps['classifier'].feature_importances_
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.barh([feature_names[i] for i in sorted_idx[:15]][::-1], importances[sorted_idx[:15]][::-1], color='#4C72B0')
plt.xlabel('Mean Decrease in Impurity')
plt.title('Top 15 Most Influential Features (Random Forest)')
plt.tight_layout()
plt.show()

## 7. Baseline Classifier Comparison
Benchmarking against baseline architectures demonstrates the robustness of Random Forest on tabular clinical data.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=12, random_state=RANDOM_STATE),
}

results = []
for name, clf in models.items():
    pipe = Pipeline([('preprocessor', build_preprocessor()), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, probs),
    })

comp_df = pd.DataFrame(results).round(4)
comp_df

## 8. Model Serialization
Save the complete preprocessor + model pipeline to `models/diabetes_model.pkl` for serving.

In [ ]:
model_out = Path('models/diabetes_model.pkl') if Path('models').exists() else Path('../models/diabetes_model.pkl')
model_out.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(rf_pipeline, model_out)
print(f'Model pipeline saved to: {model_out.resolve()}')